# Notebook 07 – Guideline JSON

Convert validated guidelines into the final **deterministic JSON repository** consumed by the adaptive website.

No LLM inference — structured mapping only.

**Input:** `validated_guidelines.csv` from Notebook 06

**Outputs:**
- `data/outputs/verified_guidelines.json`
- `data/outputs/verified_guidelines_pretty.json`
- `reports/Guideline_JSON/guideline_catalog.xlsx`


In [1]:
import json
import logging
import sys
from pathlib import Path

import pandas as pd


def _bootstrap_project() -> Path:
    search_from = Path.cwd().resolve()
    candidates = [search_from, *search_from.parents]
    nested_root = search_from / "EvidenceBasedAdaptiveUI"
    if (nested_root / "src" / "config.py").exists():
        candidates.insert(0, nested_root)
    for candidate in candidates:
        if (candidate / "src" / "config.py").exists():
            root = str(candidate)
            if root not in sys.path:
                sys.path.insert(0, root)
            return candidate
    raise FileNotFoundError("Could not find project root containing src/config.py.")


PROJECT_ROOT = _bootstrap_project()

from src.evidence_engine.json_pipeline import run_guideline_json_pipeline
from src.utils.notebook import setup_notebook

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

PATHS, REPORTS = setup_notebook("Guideline_JSON")
OUTPUT_DIR = PATHS.data_outputs
VALIDATED_PATH = PATHS.reports / "Guideline_Validation" / "validated_guidelines.csv"

print(f"Input: {VALIDATED_PATH}")
print(f"JSON output: {OUTPUT_DIR}")
print(f"Catalog output: {REPORTS}")


Input: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Guideline_Validation/validated_guidelines.csv
JSON output: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/data/outputs
Catalog output: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Guideline_JSON


## Load Validated Guidelines


In [2]:
if not VALIDATED_PATH.exists():
    raise FileNotFoundError(
        f"Missing {VALIDATED_PATH.name}. Run Notebook 06 first."
    )

validated = pd.read_csv(VALIDATED_PATH)
print(f"Validated guidelines loaded: {len(validated)} rows")
display(validated[[
    "Predictor", "UI_Element", "Guideline_Confidence_Score", "Guideline_Strength"
]].head(10))


Validated guidelines loaded: 100 rows


,Predictor,UI_Element,Guideline_Confidence_Score,Guideline_Strength
0,Agreeableness_Level,mobile_sticky_header,0.783911,Very Strong
1,primary_persona,mobile_sticky_header,0.741986,Very Strong
2,current_mood,desktop_persistent_filters,0.732949,Very Strong
3,Openness_Level,mobile_category_display,0.731728,Very Strong
4,Agreeableness_Level,mobile_review_display,0.701535,Very Strong
5,Conscientiousness_Level,mobile_whitespace,0.698410,Strong
6,primary_persona,mobile_price_display,0.694557,Strong
7,current_mood,desktop_image_text_ratio,0.693115,Strong
8,current_mood,mobile_filter_location,0.685664,Strong
9,Neuroticism_Level,desktop_persistent_filters,0.683598,Strong


## Build JSON Repository


In [3]:
result = run_guideline_json_pipeline(
    PROJECT_ROOT,
    OUTPUT_DIR,
    REPORTS,
)

rules = result.rules
validation = result.validation
summary = result.summary

print(f"Rules generated: {len(rules)}")
print(f"Validation passed: {validation.is_valid}")
if validation.warnings:
    print(f"Warnings: {len(validation.warnings)}")


INFO: Exported verified guideline repository to /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/data/outputs


Rules generated: 100
Validation passed: True


## Validation Report


In [4]:
checks = {
    "Duplicate rule IDs": validation.duplicate_rule_ids,
    "Missing fields": validation.missing_fields,
    "Invalid values": validation.invalid_values,
    "Warnings": validation.warnings,
}

for label, items in checks.items():
    if items:
        print(f"{label}:")
        for item in items[:10]:
            print(f"  - {item}")
        if len(items) > 10:
            print(f"  ... and {len(items) - 10} more")
    else:
        print(f"{label}: none")


Duplicate rule IDs: none
Missing fields: none
Invalid values: none
Warnings: none


## Sample JSON Rules


In [5]:
for rule in rules[:3]:
    print(json.dumps(rule, indent=2))
    print("-" * 60)


{
  "rule_id": 1,
  "conditions": {
    "persona": null,
    "emotion": null,
    "device": null,
    "personality": {
      "Agreeableness": "High",
      "Openness": "High"
    }
  },
  "adaptations": {
    "sticky_header": "Yes (Header stays at top while scrolling)",
    "whitespace": "Balanced"
  },
  "support": 0.16,
  "confidence": 0.785714,
  "lift": 1.289075,
  "feature_importance": {
    "agreeableness": 0.206249,
    "persona": 0.20069,
    "mood": 0.201158,
    "neuroticism": 0.210107,
    "extraversion": 0.181797
  },
  "guideline_score": 0.783911
}
------------------------------------------------------------
{
  "rule_id": 2,
  "conditions": {
    "persona": "Deal Hunter",
    "emotion": null,
    "device": null,
    "personality": {}
  },
  "adaptations": {
    "sticky_header": "Yes (Header stays at top while scrolling)"
  },
  "support": 0.135,
  "confidence": 0.875,
  "lift": 1.356589,
  "feature_importance": {
    "agreeableness": 0.206249,
    "persona": 0.20069,
    

## Final Summary


In [6]:
print(f"Total guidelines: {summary['total_guidelines']}")
print(f"Average confidence: {summary['average_confidence']:.3f}")
print(f"Average support: {summary['average_support']:.3f}")
print(f"Average guideline score: {summary['average_guideline_score']:.3f}")
print("\nExports:")
for name, path in result.export_paths.items():
    print(f"- {name}: {path}")
print("\nRepository successfully generated.")


Total guidelines: 100
Average confidence: 0.761
Average support: 0.145
Average guideline score: 0.614

Exports:
- verified_guidelines_json: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/data/outputs/verified_guidelines.json
- verified_guidelines_pretty_json: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/data/outputs/verified_guidelines_pretty.json
- guideline_catalog_xlsx: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Guideline_JSON/guideline_catalog.xlsx

Repository successfully generated.
